# 第3章 ハンズオン：GPTQ で自分で 4bit 量子化

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shintaro-Abe/llm-quantization-dojo/blob/main/notebooks/03_gptq.ipynb)

| 項目 | 内容 |
| :-- | :-- |
| 所要時間 | 約 15〜30 分（無料 Colab T4・初回はモデルDLと量子化を含む） |
| 必要ランタイム | 無料 Colab の **GPU（T4）** ランタイム（**GPTQ量子化はGPU必須**） |
| 既定モデル | `HuggingFaceTB/SmolLM2-1.7B`（Apache-2.0 / 非ゲート / safetensors） |
| やること | 校正データ用意 → GPTQ で 4bit 量子化 → 保存 → 推論（前後比較） |
| 量子化バックエンド | `gptqmodel`（AutoGPTQ は非推奨） |
| 最終確認日 | 2026-06-21 |

!!! note
    **このNotebookの検証状況**：作者により**静的検証（nbformat妥当・各コードセルの構文）＋ロジック/公式ドキュメント整合**まで実施済みです。
    **GPTQ量子化はGPU必須**のため、**T4実機での「Run all」完走は、あなたの Colab 環境で確認してください**。

## このNotebookでやること

第2章は CPU 配布向け（GGUF）でした。第3章は **GPU で精度を保ったまま 4bit にする GPTQ** を、自分の手で行います。

1. **校正データを用意** … GPTQ は代表テキストを流して誤差最小化する（だから校正データが要る）
2. **4bit 量子化** … `GPTQConfig` で SmolLM2-1.7B を量子化
3. **保存してサイズを見る** … 量子化前後の容量を比較
4. **推論** … 量子化モデルで生成して確認

!!! warning "ランタイムは GPU（T4）にしてください"
    メニュー →「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。GPTQ の量子化には GPU が必要です。

## セットアップ

現行の推奨バックエンド **`gptqmodel`** と Transformers/Optimum を導入します（`AutoGPTQ` は非推奨・使いません）。

In [ ]:
# 量子化に必要なライブラリ（gptqmodel は現行の推奨バックエンド。AutoGPTQ は使わない）
!pip install -q -U accelerate optimum transformers
# --no-build-isolation: gptqmodel はインストール時に既存の torch を参照してビルドするため、
#   ビルド分離を無効化して環境の torch を使わせる（公式手順）
!pip install -q gptqmodel --no-build-isolation

In [ ]:
# GPU 確認（GPTQ量子化には GPU が必須）＋ 再現性のため seed 固定
!nvidia-smi -L
import torch
from transformers import set_seed
assert torch.cuda.is_available(), 'GPUが見つかりません。ランタイムのタイプを T4 GPU に変更してください。'
set_seed(42)
print('CUDA OK:', torch.cuda.get_device_name(0))

## ステップ1：校正データ（calibration data）を用意する

GPTQ は「代表的な入力テキストを実際に流し、出力の誤差が最小になるよう量子化」します。そのための短いテキストを用意します。

本Notebookは外部依存ゼロのため**直書き**します。**実務では `c4` や `wikitext` などの公開データセットを多めに**使うのが一般的です（[座学の発展コラム参照](https://shintaro-abe.github.io/llm-quantization-dojo/chapters/03-gptq/)）。

In [ ]:
# 校正用テキスト（list[str]）。GPTQConfig は文字列のリストを直接受け取れる
calibration_data = [
    'The capital of France is Paris, a major European city known for art and culture.',
    'Machine learning models can be compressed through quantization to run efficiently.',
    'Python is a popular programming language for data science and artificial intelligence.',
    'The sun rises in the east and sets in the west every single day.',
    'Large language models are trained on vast amounts of text from the internet.',
    'Water boils at one hundred degrees Celsius at standard atmospheric pressure.',
    'Quantization reduces the memory footprint of neural networks with minimal accuracy loss.',
    'A balanced diet and regular exercise contribute to long-term health and well-being.',
    'The library was quiet except for the soft sound of turning pages.',
    'Renewable energy sources include solar, wind, and hydroelectric power.',
    'Reading books regularly can improve vocabulary and critical thinking skills.',
    'The recipe calls for flour, sugar, eggs, and a pinch of salt.',
    'Mountains are formed over millions of years by the movement of tectonic plates.',
    'Good software design favors simplicity, readability, and maintainability.',
    'The train departs from the station at exactly nine o clock in the morning.',
    'Photosynthesis allows plants to convert sunlight into chemical energy.',
    'Clear communication is essential for effective teamwork and collaboration.',
    'The ancient castle stood on a hill overlooking the peaceful village below.',
]
print(f'{len(calibration_data)} 件の校正テキストを用意しました')

## ステップ2：GPTQ で 4bit 量子化する

`GPTQConfig` に bit数・校正データ・トークナイザを渡し、`from_pretrained` のときに量子化を実行します。
`group_size=128` は精度とサイズのバランスの定番です。**量子化には数分かかります**（GPUで校正を回すため）。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig

MODEL = 'HuggingFaceTB/SmolLM2-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

gptq_config = GPTQConfig(bits=4, dataset=calibration_data, tokenizer=tokenizer, group_size=128)

# device_map='auto' で GPU を使って量子化する（数分かかります）
# メモリ不足(OOM)が出たら、max_memory で GPU/CPU の配分を指定する。例:
#   AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto',
#       max_memory={0: '14GiB', 'cpu': '8GiB'}, quantization_config=gptq_config)
quantized_model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map='auto', quantization_config=gptq_config,
)
print('量子化が完了しました')

## ステップ3：保存して、量子化前後のサイズを比べる

量子化したモデルを保存し、**元（fp16）と GPTQ（4bit）のファイルサイズ**を比べます。

In [ ]:
import os, glob
from huggingface_hub import snapshot_download

GPTQ_DIR = 'SmolLM2-1.7B-GPTQ-4bit'
# device_map を使ったので、保存前に CPU へ移してから保存する
quantized_model.to('cpu')
quantized_model.save_pretrained(GPTQ_DIR)
tokenizer.save_pretrained(GPTQ_DIR)

def safetensors_bytes(d):
    return sum(os.path.getsize(f) for f in glob.glob(os.path.join(d, '**', '*.safetensors'), recursive=True))

orig_dir = snapshot_download(MODEL)  # 既にキャッシュ済み（再DLなし）
orig_bytes = safetensors_bytes(orig_dir)
gptq_bytes = safetensors_bytes(GPTQ_DIR)

print(f'元(fp16)  : {orig_bytes/1024**3:.2f} GiB')
print(f'GPTQ(4bit): {gptq_bytes/1024**3:.2f} GiB')
print(f'縮小率 = {100*(1-gptq_bytes/orig_bytes):.1f}%  （{orig_bytes/max(gptq_bytes,1):.2f} 倍小さい）')

## ステップ4：量子化したモデルで推論する

保存した GPTQ モデルを読み込み直して、実際に生成させます（保存→ロードの流れも体験）。

In [ ]:
# 保存した4bitモデルを読み込んで推論
gptq_model = AutoModelForCausalLM.from_pretrained(GPTQ_DIR, device_map='auto')

prompt = 'The capital of France is'
inputs = tokenizer(prompt, return_tensors='pt').to(gptq_model.device)
out = gptq_model.generate(**inputs, max_new_tokens=40, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## まとめと振り返り

おつかれさまでした。この章で体験したこと：

- **校正データ**を使って、自分で **GPTQ 4bit 量子化**を実行した
- 量子化で**ファイルサイズが約1/3前後**に縮み、それでも文章が成立することを確認した
- GPTQ が「なぜ校正データを使うのか（誤差最小化）」を体感した

### 理解度を確認する

- GPTQ が「なぜ校正データを使うのか」を自分の言葉で説明できる
- 量子化の前後で「サイズ・出力」がどう変わったかを説明できる
- 第1章(NF4/QLoRA)・第2章(GGUF)・第3章(GPTQ)の使い分けを言える

!!! tip "発展"
    - 校正データは**公開データセット（`c4`/`wikitext` 等）を多めに**使うと、より安定した量子化になります。
    - 量子化は時間がかかるので、**まず Hugging Face Hub に既存の GPTQ 量子化済みモデルが無いか確認**するのも実務の定石です。

### 公式情報で裏取りする

- [第3章 公式リンク集（references）](https://shintaro-abe.github.io/llm-quantization-dojo/chapters/03-gptq/references/)